# Sparse Attention Visualizer: MiniMax-Style on CPU

Bu notebook'ta:
1. Sparse attention'ın nasıl çalıştığını göreceğiz
2. Full vs Random vs Top-k attention'ı karşılaştıracağız
3. Farklı k değerlerinin accuracy-speed tradeoff'unu inceleyeceğiz
4. Kendi cümlelerimizle deneyeceğiz

**GPU gerekmez!** Her şey CPU'da çalışır.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from models import build_model, FullAttention, TopKSparseAttention, RandomSparseAttention
from utils import SimpleTokenizer, get_single_example_attention
from attention_viz import (
    plot_attention_heatmap,
    plot_token_selection,
    plot_sparsity_pattern,
    plot_attention_comparison,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print(f'PyTorch: {torch.__version__}')
print(f'Device: CPU (by design!)')

## 1. Attention Mekanizması Nedir?

Attention, bir transformer'ın her kelimeyi diğer kelimelere göre "tartması"dır.

**Problem:** N token için N×N boyutunda bir matris hesaplanır → O(N²) complexity.

**Çözüm:** Sparse attention — sadece önemli token'lara bak!

In [ ]:
# Kendi cümleni dene!
TEXT = "I love this movie, it's absolutely fantastic and wonderful!"

# Tokenizer
tokenizer = SimpleTokenizer(vocab_size=500)
tokenizer.build_vocab([TEXT])
vocab_size = tokenizer.vocab_len()

tokens = tokenizer.decode(tokenizer.encode(TEXT))
print(f'Tokens: {tokens}')
print(f'Vocab size: {vocab_size}')

## 2. Full Attention vs Top-k Sparse Attention

In [ ]:
# Build models
model_full = build_model(vocab_size, attention_type='full', max_seq_len=64)
model_topk = build_model(vocab_size, attention_type='topk', k=5, max_seq_len=64)
model_random = build_model(vocab_size, attention_type='random', k=5, max_seq_len=64)

# Share weights so comparison is fair
model_topk.load_state_dict(model_full.state_dict(), strict=False)
model_random.load_state_dict(model_full.state_dict(), strict=False)

# Get attention maps
result_full = get_single_example_attention(model_full, TEXT, tokenizer, max_len=32)
result_topk = get_single_example_attention(model_topk, TEXT, tokenizer, max_len=32)
result_random = get_single_example_attention(model_random, TEXT, tokenizer, max_len=32)

print(f'Tokens: {result_full["tokens"]}')
print(f'Sequence length: {result_full["seq_len"]}')

In [ ]:
# Side-by-side comparison
results_dict = {
    'full': result_full,
    'topk': result_topk,
    'random': result_random,
}
plot_attention_comparison(results_dict, result_full['tokens'])

## 3. Hangi Token'lara Bakılıyor?

In [ ]:
# Token selection: which tokens does each query attend to?
for i, token in enumerate(result_full['tokens'][:5]):
    print(f'\nQuery: "{token}"')
    plot_token_selection(
        result_topk['attention_weights'][0],
        result_topk['tokens'],
        query_idx=i,
        title=f'Top-k Attention from "{token}"',
    )

## 4. Sparsity Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, result) in zip(axes, results_dict.items()):
    attn = result['attention_weights'][0][0]  # layer 0, head 0
    n = result['seq_len']
    binary = (attn[:n, :n] > 1e-6).astype(float)
    sparsity = 1.0 - binary.mean()
    
    ax.imshow(binary, cmap='Greys', aspect='equal')
    label = {'full': 'Full', 'topk': 'Top-k', 'random': 'Random'}[name]
    ax.set_title(f'{label}\nSparsity: {sparsity:.1%}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')

plt.suptitle('Sparsity Patterns (white = attended, black = ignored)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. k Değerinin Etkisi

In [ ]:
k_values = [2, 3, 5, 8, len(result_full['tokens'])]

fig, axes = plt.subplots(1, len(k_values), figsize=(5 * len(k_values), 4))

for ax, k in zip(axes, k_values):
    model_k = build_model(vocab_size, attention_type='topk', k=k, max_seq_len=64)
    model_k.load_state_dict(model_full.state_dict(), strict=False)
    res = get_single_example_attention(model_k, TEXT, tokenizer, max_len=32)
    
    attn = res['attention_weights'][0][0]  # layer 0, head 0
    n = res['seq_len']
    
    sns.heatmap(attn[:n, :n], cmap='YlOrRd', vmin=0, ax=ax, square=True, cbar=False)
    ax.set_title(f'k = {k}', fontsize=13, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Effect of k on Attention Pattern', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Kendi Cümleni Dene!

Aşağıdaki `YOUR_TEXT` değişkenini değiştir ve hücreyi çalıştır.

In [ ]:
YOUR_TEXT = "Tech stocks surged today as AI companies reported record profits"
YOUR_K = 5

# Rebuild tokenizer for new text
tok = SimpleTokenizer(vocab_size=500)
tok.build_vocab([YOUR_TEXT])

m_full = build_model(tok.vocab_len(), attention_type='full', max_seq_len=64)
m_topk = build_model(tok.vocab_len(), attention_type='topk', k=YOUR_K, max_seq_len=64)
m_topk.load_state_dict(m_full.state_dict(), strict=False)

r_full = get_single_example_attention(m_full, YOUR_TEXT, tok, max_len=32)
r_topk = get_single_example_attention(m_topk, YOUR_TEXT, tok, max_len=32)

plot_attention_comparison(
    {'full': r_full, 'topk': r_topk},
    r_full['tokens'],
)

print(f'\nTokens: {r_full["tokens"]}')
print(f'k = {YOUR_K} → Only top {YOUR_K} tokens attended per query')

## 7. Full Training & Evaluation

Daha kapsamlı sonuçlar için `main.py`'ı çalıştır:

```bash
python main.py --dataset synthetic --samples 500 --epochs 3 --k-values 5 10 20 50
```

Ya da IMDB ile:

```bash
python main.py --dataset imdb --samples 1000 --epochs 5 --k-values 5 10 20 50 100
```